---
>「誰もが、自分自身の法則を見つけなければならない」
>
> スティーブ・ウォズニアック

---

# 機械学習システムの運用 — 共有GPUサーバでの実験から本番学習まで

本テキストでは、研究室の共有GPUサーバ上で**安全かつ再現可能に**機械学習実験を回すための技術と知識を体系的に学ぶ

機械学習の研究・開発では、モデルの設計やアルゴリズムの理解だけでなく、**実験環境の構築・管理・運用**が成果を左右する

コードが正しくても、環境が壊れていれば実験は再現できない
学習が途中で落ちたとき、復旧手段がなければ数日分の計算が無駄になる

本テキストでは以下を扱う：

1. **リモート開発環境の構築**: VSCode Remote-SSHによるサーバ接続
2. **セッション管理**: tmuxによる長時間学習の保護
3. **仮想環境とパッケージ管理**: uvによる高速・軽量な環境構築
4. **実験管理**: Weights & Biases（wandb）によるログ収集と可視化
5. **防御的な学習実行**: Dry RunとCheckpointによる事故防止
6. **GPUリソース管理**: 共有サーバでのマナーとモニタリング
7. **データ管理とストレージ**: NFS問題とローカルディスクの使い分け
8. **実験の再現性**: シード固定、設定管理、バージョン管理
9. **本番学習のワークフロー**: 実践的なMLOpsの基礎
10. **トラブルシューティング**: よくある障害と対処法

# 参考文献

- Weights & Biases Documentation. https://docs.wandb.ai/
- uv: An extremely fast Python package installer and resolver. https://docs.astral.sh/uv/
- tmux Wiki. https://github.com/tmux/tmux/wiki
- VSCode Remote Development. https://code.visualstudio.com/docs/remote/ssh
- PyTorch Documentation. https://pytorch.org/docs/stable/
- Sculley, D. et al. (2015). "Hidden Technical Debt in Machine Learning Systems." NeurIPS 2015.
- NVIDIA Systems Management Interface. https://developer.nvidia.com/nvidia-system-management-interface
- Google. "Rules of Machine Learning: Best Practices for ML Engineering." https://developers.google.com/machine-learning/guides/rules-of-ml
- Huyen, C. (2022). "Designing Machine Learning Systems." O'Reilly Media.
- Paleyes, A. et al. (2022). "Challenges in Deploying Machine Learning: a Survey of Case Studies." ACM Computing Surveys.


---

# この講義でできるようになること

- 研究室の共有GPUサーバで、安全に機械学習実験を回せるようになる
- `VSCode Remote-SSH`、`tmux`、`uv`、`wandb` を使った標準的な開発フローを一通り実践できるようになる
- `Dry Run` と `Checkpoint` を使って、よくある事故を事前に潰せるようになる

# 全体像

まず、本テキストで扱う作業フロー全体を俯瞰する

1. VSCodeからサーバに入る
2. `tmux` で安全な作業セッションを作る
3. `uv` で仮想環境を作る
4. `wandb` を連携して実験ログを取る
5. `train.py` を作って `Dry Run` を回す
6. 本番学習を流し、必要なら途中再開する
7. 学習率を変えて、WandB上で結果を比較する

この流れは、個人の研究プロジェクトでもチーム開発でも共通する**基本パターン**である

以降のセクションで、各ステップを順に解説する

---

## 最重要ルール: データ配置

**学習用の重いデータをホームディレクトリ `~/` に置いてはいけない**

ホームディレクトリはNFS（Network File System）であり、大量の読み込みを発生させると**研究室全体のネットワークを重くする**

本番では、`/storage/...` のような各サーバのローカル大容量ディスクを使うこと

今回だけはMNISTが小さいため、特例として `root='./data'` を使う

### なぜNFSが問題になるのか

NFSはネットワーク越しにファイルを共有する仕組みである
ホームディレクトリは通常NFSでマウントされており、どのサーバからログインしても同じファイルが見える

しかし、機械学習の学習ループではデータを**毎エポック繰り返し読み込む**
ImageNetのような大規模データセットをNFS上に置くと、ネットワーク帯域を独占し、他のユーザのSSH接続すら遅くなる

| 配置場所 | 速度 | 他ユーザへの影響 | 用途 |
| --- | --- | --- | --- |
| `~/`（NFS） | 遅い | 大きい | コード、設定ファイル |
| `/storage/`（ローカルSSD） | 速い | なし | データセット、チェックポイント |

---

# Step 1. VSCode Remote-SSHでサーバに入る

## 目的

手元のPCから、サーバ上のコードを**直接編集・実行**できる状態を作る

## なぜVSCode Remote-SSHか

従来のリモート開発では、ターミナルで `ssh` → `vim` でコード編集という流れが一般的だった
しかし、この方法では以下の問題がある：

- エディタの補完・リント機能が使えない
- ファイルツリーの俯瞰が困難
- デバッガの利用が難しい

VSCode Remote-SSHは、ローカルのVSCodeからSSH経由でサーバに接続し、**まるでローカルで作業しているかのように**リモートのファイルを編集・実行できる

## 手順

1. VSCodeの拡張機能 `Remote - SSH` をインストールする
2. VSCode左下の接続メニューから `Connect to Host...` を選ぶ
3. `(username)@サーバーのIP` を入力して接続する
4. 接続後、VSCode内のターミナルを開いて操作する

## SSH設定ファイルの活用

毎回IPアドレスとユーザ名を入力するのは手間である
`~/.ssh/config` に設定を書いておくと、ホスト名だけで接続できる

In [ ]:
# ~/.ssh/config の設定例（ローカルPCで編集する）
# このセルは実行不要: 設定ファイルの書き方を示すための参考コード

ssh_config = """
Host lab-gpu1
    HostName 192.168.1.100
    User your_username
    IdentityFile ~/.ssh/id_ed25519
    ForwardAgent yes

Host lab-gpu2
    HostName 192.168.1.101
    User your_username
    IdentityFile ~/.ssh/id_ed25519
"""
print(ssh_config)

## 確認事項

- サーバ上のターミナルが開けること
- サーバのホームディレクトリを見られること
- `python --version` でPythonが使えること

---

# Step 2. `tmux` でセッションを守る

## なぜ必要か

SSH接続は切れる
接続が切れると、普通のターミナルで動かしていた学習も**一緒に死ぬ**

長時間学習では `tmux`（terminal multiplexer）の使用が**必須**である

## tmuxの仕組み

`tmux` はサーバ上でバックグラウンドに常駐する**仮想ターミナル**である

```
[ローカルPC] --SSH--> [サーバ] --tmux--> [仮想ターミナル]
                         |                    |
                     SSH切断!              学習継続!
```

SSHが切れても `tmux` のセッションはサーバ上で生き続ける
再接続後に `tmux attach` するだけで、切断前の画面に復帰できる

## 最低限のコマンド

In [ ]:
# tmux の基本操作（ターミナルで実行する）
# このセルは実行不要: コマンド例を示すための参考コード

tmux_commands = """
# セッションの作成（名前付き）
tmux new -s ml_expr

# セッションから抜ける（デタッチ）
# [Ctrl + b] のあとに [d]

# セッション一覧の確認
tmux ls

# セッションへ復帰（アタッチ）
tmux attach -t ml_expr

# セッションを完全に終える
exit
"""
print(tmux_commands)

## tmuxの便利な操作

| 操作 | キーバインド | 説明 |
| --- | --- | --- |
| デタッチ | `Ctrl+b` → `d` | セッションを残して抜ける |
| ウィンドウ作成 | `Ctrl+b` → `c` | 新しいタブを作る |
| ウィンドウ切替 | `Ctrl+b` → `n` / `p` | 次/前のタブに移動 |
| 画面分割（横） | `Ctrl+b` → `%` | 画面を左右に分割 |
| 画面分割（縦） | `Ctrl+b` → `"` | 画面を上下に分割 |
| ペイン切替 | `Ctrl+b` → 矢印キー | 分割画面間を移動 |
| スクロール | `Ctrl+b` → `[` | スクロールモードに入る（`q`で抜ける） |

## 実践ルール

- 学習を始める前に**必ず** `tmux new -s ml_expr` を実行する
- 切断しても、学習は `tmux` の中で動き続ける
- 作業終了後は不要なセッションを `exit` で閉じる
- セッション名は実験内容がわかる名前にする（例: `resnet_lr001`、`bert_finetune`）

## byobu

byobuは、tmuxもしくはscreenのラッパーとして動作するツールである

様々な便利なデフォルト設定が盛り込まれており、ステータスバーが標準でついてくるため、初心者にはお勧めである

---

# Step 3. `uv` で仮想環境を作る

## なぜ仮想環境が必要か

共有サーバでは複数のユーザが同じマシンを使う
システムのPython環境にパッケージを直接インストールすると、バージョンの競合が発生し、**他のユーザの実験を壊す**可能性がある

仮想環境はプロジェクトごとに独立したPython環境を作る仕組みである

## なぜ `uv` を使うのか

Pythonのパッケージ管理ツールは多数あるが、`uv` には以下の利点がある：

| ツール | 速度 | 環境作成 | 依存解決 | 特徴 |
| --- | --- | --- | --- | --- |
| `pip` + `venv` | 遅い | 手動 | 基本的 | 標準付属 |
| `conda` | 遅い | 自動 | 強力 | 重い、ライセンス問題 |
| `poetry` | 普通 | 自動 | 強力 | 設定が複雑 |
| **`uv`** | **極めて速い** | **自動** | **強力** | **Rust製、軽量** |

`uv` は `pip` の**10〜100倍高速**であり、共有サーバの環境を汚さずに済む

## 作業の流れ

In [ ]:
# uv のセットアップ手順（ターミナルで実行する）
# このセルは実行不要: コマンド例を示すための参考コード

setup_commands = """
# 1. プロジェクトディレクトリの作成
mkdir -p educ_ml
cd educ_ml

# 2. uv のインストール
curl -LsSf https://astral.sh/uv/install.sh | sh

# 3. PATHの設定
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.zshrc
# 注: 上記は zsh ユーザー向け。bash ユーザーは ~/.bashrc に置き換えること
source ~/.zshrc

# 4. バージョン確認
uv --version

# 5. 仮想環境の作成と有効化
uv venv
source .venv/bin/activate

# 6. パッケージのインストール
uv pip install torch torchvision wandb
"""
print(setup_commands)

## `uv` の追加コマンド

```bash
# インストール済みパッケージの確認
uv pip list

# requirements.txt からの一括インストール
uv pip install -r requirements.txt

# 現在の環境をrequirements.txtに書き出す
uv pip freeze > requirements.txt

# 仮想環境の無効化
deactivate
```

## 確認事項

- ターミナルの先頭に `(.venv)` が出ていること
- `python -c "import torch; print(torch.__version__)"` が動くこと
- `python -c "import wandb; print(wandb.__version__)"` が動くこと

---

# Step 4. `wandb` を連携する

## なぜ実験管理が必要か

ターミナルの数字だけでは、学習が良いのか悪いのか判断しにくい

`wandb`（Weights & Biases）を使うと、Loss、学習率、GPU使用状況などを**リアルタイムでブラウザ上に可視化**できる

## wandbが解決する問題

| 問題 | wandbなし | wandbあり |
| --- | --- | --- |
| 学習曲線の確認 | ターミナルの数字を目視 | グラフで自動可視化 |
| 実験の比較 | メモ帳に手書き | ダッシュボードで並列比較 |
| ハイパラの記録 | ファイル名に埋め込み | 自動的にメタデータ保存 |
| 再現性 | 「あの実験のパラメータ何だっけ」 | 全設定が記録済み |

## 手順

1. [wandb.ai](https://wandb.ai) でアカウントを作る
2. ターミナルでログインする

In [ ]:
# wandb のセットアップ（ターミナルで実行する）
# このセルは実行不要: コマンド例を示すための参考コード

wandb_setup = """
# wandb にログイン（APIキーを入力する）
wandb login

# APIキーはブラウザで https://wandb.ai/authorize から取得できる
"""
print(wandb_setup)

## wandbの基本的な使い方（コード内）

```python
import wandb

# 実験の初期化
wandb.init(
    project="my-project",       # プロジェクト名
    name="experiment-001",       # この実験の名前（省略可）
    config={                     # ハイパーパラメータの記録
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 64,
        "model": "AutoEncoder",
    }
)

# 学習ループ内でメトリクスを記録
wandb.log({
    "epoch": epoch,
    "train_loss": loss_value,
    "learning_rate": current_lr,
})

# 実験の終了
wandb.finish()
```

## 確認事項

- APIキーの入力が通ること
- 学習開始後にWandBの実験ページURLが出ること

---

# Step 5. 実装・実行・監視を一通り回す

ここからが本番である
コードを書き、学習を回し、結果を監視するまでの一連の流れを実践する

## 5-1. まず理解してほしい2つの防御技術

### Dry Run（ドライラン）

最初の1バッチだけ流して、タイポ、次元不一致、メモリエラーが出ないかを**即座に確認**するための機能である

10時間の学習を流す前に、**数秒で壊れていないことを確認する**

Dry Runで検出できる典型的なバグ：

| バグの種類 | 例 | Dry Runなしだと |
| --- | --- | --- |
| テンソル次元不一致 | `view(-1, 784)` の784が間違い | 学習開始直後にクラッシュ |
| GPU OOM | バッチサイズが大きすぎる | 数分後にOOMで死ぬ |
| import漏れ | `import wandb` 忘れ | 初回ログ時にNameError |
| データパス間違い | `root='./dat'` のtypo | データロード時にFileNotFoundError |

### Checkpoint（チェックポイント）

OOMや切断で学習が止まっても、**途中のEpochから再開**できるようにするための保存機能である

長時間学習では**必須**である

Checkpointに保存すべき情報：

- `model.state_dict()`: モデルの重み
- `optimizer.state_dict()`: オプティマイザの状態（モメンタムなど）
- `epoch`: 現在のエポック番号
- `loss`: 直近のロス値（デバッグ用）

## 5-2. `train.py` の全体像

以下のコードを `train.py` として保存する

コードの各パートの役割を理解しながら読み進めること

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import wandb
import os

# --- 実験用設定 ---
# Dry Run: 本番流しの前に、エラーが出ないか「1バッチだけ」回して強制終了するテストモード
DRY_RUN = True
# Resume: プログラムが落ちた場合に、保存された重みから途中のエポックを再開するか
RESUME_TRAINING = False
CHECKPOINT_PATH = "autoencoder_latest.pth"

# 1. wandbの初期化（実験の記録開始）
wandb.init(project="b4-training-autoencoder", config={
    "learning_rate": 0.001,
    "epochs": 5 if not DRY_RUN else 1,
    "batch_size": 64,
})
config = wandb.config

# 2. デバイスの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. データの準備
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

# 4. モデルの定義
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 32), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 128), nn.ReLU(),
            nn.Linear(128, 28 * 28), nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = AutoEncoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

# --- チェックポイントからの再開 ---
start_epoch = 0
if RESUME_TRAINING and os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"チェックポイントを読み込みました Epoch {start_epoch+1} から再開します")

# 5. 学習ループ
model.train()
for epoch in range(start_epoch, config.epochs):
    epoch_loss = 0.0
    for batch_idx, (images, _) in enumerate(train_loader):
        images = images.view(-1, 28 * 28).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if DRY_RUN:
            print("[DRY RUN] 1バッチ目の計算が正常に完了しました")
            break

    avg_loss = epoch_loss / (1 if DRY_RUN else len(train_loader))
    print(f"Epoch [{epoch+1}/{config.epochs}], Loss: {avg_loss:.4f}")

    current_lr = optimizer.param_groups[0]['lr']
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "learning_rate": current_lr
    })

    if not DRY_RUN:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, CHECKPOINT_PATH)
        print(f"Epoch {epoch+1} のモデル状態を '{CHECKPOINT_PATH}' に保存しました")

print("学習プロセス完了")
wandb.finish()

## 5-3. コードの構造解説

上記のコードは以下の5つのブロックから構成されている

### ブロック1: 設定（Configuration）

```python
DRY_RUN = True
RESUME_TRAINING = False
CHECKPOINT_PATH = "autoencoder_latest.pth"
```

実験の動作を制御するフラグである
本番学習では `DRY_RUN = False` に変更する
学習が中断された場合は `RESUME_TRAINING = True` にして再開する

### ブロック2: wandb初期化

```python
wandb.init(project="b4-training-autoencoder", config={...})
```

`project` でプロジェクト名を指定し、`config` でハイパーパラメータを記録する
ここで記録した `config` は後からWandBダッシュボード上で確認・比較できる

### ブロック3: データとモデル

MNISTデータセットを読み込み、シンプルなAutoEncoderを定義する
AutoEncoderは入力画像を圧縮（encode）してから復元（decode）するネットワークである

### ブロック4: チェックポイント復元

```python
if RESUME_TRAINING and os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
```

`RESUME_TRAINING = True` かつチェックポイントファイルが存在する場合のみ、保存済みの状態を復元する

### ブロック5: 学習ループ

各エポックでデータを1周し、バッチごとに順伝播→損失計算→逆伝播→重み更新を繰り返す
`DRY_RUN = True` の場合は1バッチで `break` して即座に終了する

## 5-4. Dry Runの実行

まず `DRY_RUN = True` の状態で実行し、エラーがないことを確認する

```bash
python train.py
```

期待される出力：
```
Using device: cuda
[DRY RUN] 1バッチ目の計算が正常に完了しました
Epoch [1/1], Loss: 0.0xxx
学習プロセス完了
```

この出力が確認できたら、コード全体が正常に動作している

## 5-5. 本番学習の実行

`train.py` の設定を以下のように変更する：

```python
DRY_RUN = False   # Dry Runを無効化
```

そして再度実行する：

```bash
python train.py
```

学習中はWandBのダッシュボードでLossの推移をリアルタイムに確認できる

## 5-6. 学習の途中再開

学習が何らかの理由で中断された場合、以下のように設定を変更して再開できる：

```python
DRY_RUN = False
RESUME_TRAINING = True   # チェックポイントからの再開を有効化
```

これにより、最後に保存されたエポックの続きから学習が再開される

---

# Step 6. GPUリソースの管理

共有サーバでGPUを使う際は、**他のユーザとリソースを分け合う**意識が不可欠である

## GPU使用状況の確認

In [ ]:
# GPU状況の確認コマンド（ターミナルで実行する）
# このセルは実行不要: コマンド例を示すための参考コード

gpu_commands = """
# 現在のGPU使用状況を確認
nvidia-smi

# 1秒ごとに自動更新で監視
watch -n 1 nvidia-smi

# 特定のGPUのみを使用するように指定
CUDA_VISIBLE_DEVICES=0 python train.py

# GPU 2番を使う場合
CUDA_VISIBLE_DEVICES=2 python train.py
"""
print(gpu_commands)

## `nvidia-smi` の読み方

```
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03   Driver Version: 535.129.03   CUDA Version: 12.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM4   On    | 00000000:07:00.0 Off |                    0 |
| N/A   35C    P0    52W / 400W |   1234MiB / 40960MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
```

| 項目 | 意味 | 確認ポイント |
| --- | --- | --- |
| Memory-Usage | GPUメモリの使用量/総量 | 空きがあるか確認 |
| GPU-Util | GPU演算ユニットの使用率 | 他ユーザが使用中か確認 |
| Pwr:Usage/Cap | 消費電力/最大電力 | 高負荷かどうかの目安 |
| Temp | GPU温度 | 80度超は危険信号 |

## 共有サーバでのマナー

1. **使う前に `nvidia-smi` で空きGPUを確認する**
2. `CUDA_VISIBLE_DEVICES` で使用するGPUを明示的に指定する
3. 学習が終わったらプロセスを確実に終了させる
4. GPUメモリを長時間占有しない（使わないなら解放する）
5. 大規模な実験を始める前にSlackやメールで共有する

## Pythonコード内でのGPU指定

In [ ]:
import torch
import os

# 方法1: 環境変数で指定（推奨）
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # GPU 0番のみ使用

# 方法2: torch.deviceで直接指定
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# GPU情報の確認
if torch.cuda.is_available():
    print(f"利用可能なGPU数: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    メモリ: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("GPUが利用できません CPUを使用します")

---

# Step 7. 実験の再現性

機械学習の実験では、**同じコードを同じ条件で実行したら同じ結果が出る**ことが極めて重要である

再現性がなければ、どの変更が性能向上に寄与したのか判断できない

## シード固定

In [ ]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    """再現性のために全ての乱数シードを固定する"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # cuDNNの非決定的アルゴリズムを無効化（速度は低下する）
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 学習開始前に呼ぶ
set_seed(42)
print("シード固定完了")

## 注意点

- `torch.backends.cudnn.deterministic = True` は速度を犠牲にして再現性を保証する
- 性能測定時は `True` にし、本番の長時間学習では `False` にして速度を優先することもある
- `DataLoader` の `num_workers > 0` の場合、`worker_init_fn` でもシードを設定する必要がある

```python
def worker_init_fn(worker_id):
    np.random.seed(42 + worker_id)

train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    worker_init_fn=worker_init_fn
)
```

## 設定管理のベストプラクティス

ハイパーパラメータをコード中にハードコーディングすると、変更履歴が追えなくなる

設定ファイルや `argparse` で外部化するのが良い：

```python
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--lr', type=float, default=0.001)
parser.add_argument('--epochs', type=int, default=10)
parser.add_argument('--batch_size', type=int, default=64)
parser.add_argument('--seed', type=int, default=42)
parser.add_argument('--dry_run', action='store_true')
parser.add_argument('--resume', action='store_true')
args = parser.parse_args()
```

これにより、コマンドラインから実験条件を変えられる：

```bash
python train.py --lr 0.0001 --epochs 20 --batch_size 128
python train.py --dry_run
python train.py --resume
```

---

# Step 8. 学習率を変えてWandBで比較する

## なぜ学習率の比較が重要か

学習率（learning rate）はニューラルネットワークの学習で**最も影響の大きいハイパーパラメータ**の一つである

- 大きすぎる → 損失が発散して学習が壊れる
- 小さすぎる → 収束が遅く、局所最適に陥りやすい
- 適切 → 効率的に最適解に近づく

## 比較実験の手順

```bash
# 実験1: lr=0.001（デフォルト）
python train.py --lr 0.001

# 実験2: lr=0.0001（小さめ）
python train.py --lr 0.0001

# 実験3: lr=0.01（大きめ）
python train.py --lr 0.01
```

WandBダッシュボード上で、3つの実験のLoss曲線を重ねて表示し、どの学習率が最も効率的に収束するかを比較する

## wandbでの比較方法

1. WandBのプロジェクトページを開く
2. 左側のサイドバーで比較したい実験にチェックを入れる
3. 「Charts」タブでLoss曲線が重ねて表示される
4. 「Table」タブで各実験の最終Lossやハイパーパラメータを一覧比較できる

### wandb Sweepによる自動探索

手動で学習率を変えるのが面倒な場合、`wandb.sweep` を使うとハイパーパラメータの自動探索ができる

```python
sweep_config = {
    'method': 'grid',          # grid, random, bayes から選択
    'metric': {
        'name': 'train_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'learning_rate': {
            'values': [0.01, 0.001, 0.0001]
        },
        'batch_size': {
            'values': [32, 64, 128]
        }
    }
}

sweep_id = wandb.sweep(sweep_config, project="b4-training-autoencoder")
wandb.agent(sweep_id, function=train, count=9)  # 9通りの組み合わせを自動実行
```

---

# Step 9. トラブルシューティング

共有GPUサーバでの機械学習実験では、様々なトラブルが発生する
ここでは代表的な問題とその対処法をまとめる

## よくあるエラーと対処法

### CUDA Out of Memory（OOM）

```
RuntimeError: CUDA out of memory. Tried to allocate 256.00 MiB
```

**原因**: GPUメモリが不足している

**対処法**:
1. バッチサイズを小さくする
2. モデルを軽量化する
3. `torch.cuda.empty_cache()` でキャッシュを解放する
4. 勾配累積（Gradient Accumulation）を使う
5. Mixed Precision Training（`torch.cuda.amp`）を使う

In [ ]:
# 勾配累積の例
accumulation_steps = 4  # 4バッチ分の勾配を貯めてから更新

# optimizer.zero_grad()  # ループの外で1回だけ
# for batch_idx, (images, labels) in enumerate(train_loader):
#     outputs = model(images)
#     loss = criterion(outputs, labels)
#     loss = loss / accumulation_steps  # 勾配をスケーリング
#     loss.backward()
#
#     if (batch_idx + 1) % accumulation_steps == 0:
#         optimizer.step()
#         optimizer.zero_grad()

print("勾配累積: 実効バッチサイズ = batch_size x accumulation_steps")
print(f"例: batch_size=16, accumulation_steps={accumulation_steps} -> 実効バッチサイズ={16 * accumulation_steps}")

In [ ]:
# Mixed Precision Trainingの例
# GPU メモリ使用量を約半分に削減できる

# from torch.cuda.amp import autocast, GradScaler
# 注意: torch.cuda.amp は PyTorch 2.4 以降で非推奨。torch.amp.autocast("cuda") と torch.amp.GradScaler("cuda") を推奨
#
# scaler = GradScaler()
#
# for images, labels in train_loader:
#     optimizer.zero_grad()
#
#     with autocast():  # float16で順伝播
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#
#     scaler.scale(loss).backward()  # スケーリングされた勾配で逆伝播
#     scaler.step(optimizer)
#     scaler.update()

print("Mixed Precision Training: float32 -> float16 で計算")
print("メモリ使用量が約半分になり、最新GPUでは速度も向上する")

### プロセスが残留してGPUメモリを占有

学習スクリプトがエラーで終了しても、GPUメモリが解放されないことがある

```bash
# GPUを使っているプロセスを確認
nvidia-smi

# 自分のプロセスを特定して終了
kill -9 <PID>

# 自分の全てのPythonプロセスを終了（注意して使うこと）
pkill -u $(whoami) python
```

### SSH接続が切れて学習が止まった

**原因**: `tmux` を使わずに学習を実行した

**対処法**:
1. `tmux` を使う習慣をつける（Step 2参照）
2. `RESUME_TRAINING = True` にしてチェックポイントから再開する

### `ModuleNotFoundError: No module named 'xxx'`

**原因**: 仮想環境が有効化されていないか、パッケージが未インストール

**対処法**:
```bash
# 仮想環境を有効化
source .venv/bin/activate

# パッケージをインストール
uv pip install <package_name>
```

### `Permission denied` でファイルが作れない

**原因**: ストレージのパーミッション問題

**対処法**:
```bash
# ディレクトリの権限を確認
ls -la /storage/

# 自分のディレクトリを作成（管理者に依頼が必要な場合もある）
mkdir -p /storage/$(whoami)/data
```

---

# Step 10. 本番運用のベストプラクティス

ここまでの内容を統合し、実際の研究プロジェクトでの運用パターンをまとめる

## プロジェクト構成の推奨

```
~/my_project/              # ホーム（NFS）: コードと設定のみ
├── train.py
├── model.py
├── dataset.py
├── config/
│   ├── default.yaml
│   └── experiment_001.yaml
├── requirements.txt
├── .venv/                 # 仮想環境
└── README.md

/storage/username/         # ローカルSSD: データとチェックポイント
├── data/
│   ├── mnist/
│   └── imagenet/
└── checkpoints/
    ├── exp001_epoch10.pth
    └── exp001_latest.pth
```

## 実験実行チェックリスト

新しい実験を始める前に、以下を確認する：

- [ ] `nvidia-smi` で空きGPUを確認した
- [ ] `tmux` セッションの中にいる
- [ ] 仮想環境が有効化されている（`(.venv)` が表示されている）
- [ ] `CUDA_VISIBLE_DEVICES` で使用GPUを指定した
- [ ] データは `/storage/` に配置した（NFSではない）
- [ ] Dry Runが通ることを確認した
- [ ] チェックポイント保存が有効になっている
- [ ] wandbの初期化が正しく設定されている
- [ ] シード値を固定した

## 実験ログの命名規則

wandbの実験名やチェックポイントファイルに統一的な命名規則を設けると、後から探しやすくなる

```
# 推奨フォーマット
{モデル名}_{データセット}_{主要パラメータ}_{日付}

# 例
resnet50_imagenet_lr0001_bs128_20260330
bert_squad_ep10_lr5e5_20260330
```

## Gitによるコード管理

実験コードは必ずGitで管理する
「あの実験のコード、どこにあったっけ」を防ぐためである

```bash
# 基本の流れ
git init
git add train.py model.py dataset.py config/
git commit -m "Initial training setup"

# 実験ごとにブランチを切ると、後から比較しやすい
git checkout -b exp/lr_search
# ... 実験 ...
git add -A && git commit -m "LR search: 0.001, 0.0001, 0.01"
```

### `.gitignore` の設定

```
# データとチェックポイントはGitに入れない
data/
*.pth
*.pt
wandb/
.venv/
__pycache__/
```

---

# 発展: MLOpsの基礎概念

ここまでの内容は、**MLOps**（Machine Learning Operations）の入口にあたる

MLOpsとは、機械学習モデルの開発から本番運用までのライフサイクル全体を管理する技術・プラクティスの総称である

## MLOpsの全体像

```
データ収集 → 前処理 → 学習 → 評価 → デプロイ → 監視 → 再学習
    ↑                                              |
    └──────────────── フィードバックループ ──────────┘
```

本テキストで扱った内容は主に「学習」フェーズの運用技術であるが、実際のプロダクションでは以下も重要になる：

| フェーズ | 技術 | ツール例 |
| --- | --- | --- |
| データ管理 | バージョニング、品質チェック | DVC、Great Expectations |
| 学習管理 | 実験追跡、ハイパラ探索 | wandb、MLflow、Optuna |
| モデル管理 | レジストリ、バージョニング | MLflow Model Registry |
| デプロイ | サービング、バッチ推論 | TorchServe、Triton、BentoML |
| 監視 | データドリフト検知、性能劣化検知 | Evidently、WhyLabs |
| パイプライン | ワークフロー自動化 | Kubeflow、Airflow、Prefect |

## 実用的なMLOps技術

### Docker によるコンテナ化

`uv` + venv による仮想環境は手軽だが、CUDA ドライバやシステムライブラリのバージョン差異は解決できない

Docker コンテナを用いることで、OS・CUDA・Python・ライブラリを含む実行環境全体を再現可能にできる

```dockerfile
FROM pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["python", "train.py"]
```

NVIDIA Container Toolkit を導入することで、コンテナ内から GPU にアクセスできる

### CI/CD for ML

GitHub Actions 等を活用し、以下を自動化できる
- コードの静的解析（ruff、mypy）
- Dry Run による訓練スクリプトの動作確認
- テストデータでのモデル精度の回帰テスト

### モデルサービング

学習済みモデルを推論 API として公開する代表的な方法
- **FastAPI + uvicorn**: Python での最小構成の REST API。プロトタイプ向き
- **TorchServe**: PyTorch 公式のモデルサービングフレームワーク。バッチ推論、モデルバージョニングに対応
- **vLLM**: LLM 特化の高速推論サーバー。PagedAttention による効率的なメモリ管理

### 本番監視（Production Monitoring）

デプロイ後のモデルは、入力データの分布変化（データドリフト）により性能が劣化する可能性がある
- **データドリフト**: 学習時と推論時のデータ分布の乖離。季節変動や外部環境変化が原因
- **コンセプトドリフト**: 入力と出力の関係性そのものが変化する現象
- Evidently、WhyLabs、NannyML 等のツールでドリフトを検知し、再学習のトリガーとして利用する

## Hidden Technical Debt in ML Systems

Sculley et al. (2015) は、機械学習システムにおける**隠れた技術的負債**を指摘した

実際のMLシステムでは、MLコード自体は全体のごく一部であり、周辺のインフラストラクチャが大部分を占める：

```
┌─────────────────────────┐
│  Configuration   Monitoring   Feature Extraction │
│  Data Collection  Data Verification  Testing     │
│  ┌───────────────────┐      │
│  │  ML Code  │   Process Management    │      │
│  └───────────────────┘      │
│  Serving Infrastructure   Machine Resource Mgmt  │
└─────────────────────────┘
```

この認識が、本テキストで「運用」を重視する理由である
優れたモデルを作っても、運用基盤がなければ価値を発揮できない

---

# まとめ

本テキストでは、機械学習実験の運用に必要な技術を一通り学んだ

| Step | 内容 | 核心 |
| --- | --- | --- |
| 1 | VSCode Remote-SSH | ローカルと同じ体験でリモート開発 |
| 2 | tmux | SSH切断から学習を守る |
| 3 | uv | 高速・軽量な仮想環境 |
| 4 | wandb | 実験ログの自動可視化・比較 |
| 5 | Dry Run + Checkpoint | 事故防止と途中再開 |
| 6 | GPU管理 | 共有リソースの適切な利用 |
| 7 | 再現性 | シード固定と設定管理 |
| 8 | 比較実験 | 学習率探索とwandb比較 |
| 9 | トラブルシューティング | OOM、残留プロセス、権限問題 |
| 10 | ベストプラクティス | プロジェクト構成とチェックリスト |

これらは研究室での日常的な実験から、将来のプロダクション開発まで通用する基礎スキルである

**良いモデルは、良い運用の上に成り立つ**

---

# 課題1（環境構築と基本実験）

以下の手順を実施し、スクリーンショットとともに報告せよ

1. VSCode Remote-SSHでGPUサーバに接続する
2. `tmux` セッションを作成する
3. `uv` で仮想環境を構築し、必要パッケージをインストールする
4. `wandb` にログインする
5. `train.py` を作成し、Dry Runを成功させる
6. 本番学習（5エポック）を実行し、WandBダッシュボードのスクリーンショットを提出する

# 課題2（学習率の比較実験）

以下の3つの学習率で実験を行い、WandB上で比較した結果をレポートせよ

- `lr = 0.01`
- `lr = 0.001`
- `lr = 0.0001`

レポートに含める内容：
1. 3つの実験のLoss曲線を重ねたグラフ（WandBのスクリーンショット）
2. 各学習率での最終Loss値の比較表
3. どの学習率が最も良かったかの考察と理由

# 課題3（チェックポイントからの再開）

以下の手順を実施し、チェックポイント機能が正しく動作することを確認せよ

1. 5エポックの学習を開始する
2. 2エポック目が完了した時点で `Ctrl+C` で強制終了する
3. `RESUME_TRAINING = True` に設定して再開する
4. 3エポック目から再開されることを確認する
5. WandB上で学習が途切れず継続していることを示すスクリーンショットを提出する